In [1]:
import pandas as pd
from datetime import timedelta

def build_customer_dataset(accounts_path, interactions_path, products_path, sales_path):
    """
    Loads raw FarmCorp data and engineers a customer-level analytical dataset.
    """
    
    # ---------------------------------------------------------
    # 1. Load Data
    # ---------------------------------------------------------
    accounts = pd.read_parquet(accounts_path)
    interactions = pd.read_parquet(interactions_path)
    # Assuming products.txt is a CSV or Tab-separated file
    products = pd.read_csv(products_path, sep='\t') 
    sales = pd.read_parquet(sales_path)

    # Ensure datetime formatting
    interactions['date'] = pd.to_datetime(interactions['date'])
    sales['date'] = pd.to_datetime(sales['date'])

    # ---------------------------------------------------------
    # 2. Compute KPIs
    # ---------------------------------------------------------

    # KPI 1: Total revenue generated (purchases * product price)
    # Merge sales with products to attach the price to each sale transaction
    sales_with_prices = sales.merge(products[['product_id', 'price']], on='product_id', how='left')
    kpi_revenue = sales_with_prices.groupby('account_id')['price'].sum().reset_index()
    kpi_revenue.rename(columns={'price': 'total_revenue_usd'}, inplace=True)

    # KPI 2: Number of sales interactions in the last 6 months
    # We use the most recent date in the interactions dataset as our 'current' reference point
    max_date = interactions['date'].max()
    six_months_ago = max_date - pd.DateOffset(months=6)
    
    recent_interactions = interactions[interactions['date'] >= six_months_ago]
    kpi_recent_interactions = recent_interactions.groupby('account_id').size().reset_index(name='interactions_last_6m')

    # KPI 3 (Custom): Positive Interaction Rate
    # (Count of positive interactions / Total interactions)
    total_interactions = interactions.groupby('account_id').size()
    positive_interactions = interactions[interactions['response'] == 'positive'].groupby('account_id').size()
    
    # Divide and fill NaNs with 0 (for users with interactions but no positive ones)
    kpi_pos_rate = (positive_interactions / total_interactions).fillna(0).reset_index(name='positive_interaction_rate')

    # KPI 4 (Custom): Product Category Diversity
    # How many distinct categories has the customer bought from?
    sales_with_categories = sales.merge(products[['product_id', 'category']], on='product_id', how='left')
    kpi_diversity = sales_with_categories.groupby('account_id')['category'].nunique().reset_index()
    kpi_diversity.rename(columns={'category': 'distinct_categories_bought'}, inplace=True)


    # ---------------------------------------------------------
    # 3. Build Final Customer-Level Dataset
    # ---------------------------------------------------------
    # Start with the accounts base table
    analytical_df = accounts.copy()

    # Sequentially left-join the KPIs to the accounts table
    kpi_dataframes = [kpi_revenue, kpi_recent_interactions, kpi_pos_rate, kpi_diversity]
    
    for kpi_df in kpi_dataframes:
        analytical_df = analytical_df.merge(kpi_df, on='account_id', how='left')

    # Fill missing values for customers who haven't had sales or interactions yet
    fill_values = {
        'total_revenue_usd': 0,
        'interactions_last_6m': 0,
        'positive_interaction_rate': 0.0,
        'distinct_categories_bought': 0
    }
    analytical_df.fillna(value=fill_values, inplace=True)

    return analytical_df

# Example execution:
# final_dataset = build_customer_dataset('accounts.parquet', 'interactions.parquet', 'products.txt', 'sales.parquet')
# print(final_dataset.head())

In [3]:
accounts_path = 'C:\\Users\\rossim\\Desktop\\venchi_project\\venchi_jMLE_candidate_pack\\data\\accounts.parquet'
interactions_path = 'C:\\Users\\rossim\\Desktop\\venchi_project\\venchi_jMLE_candidate_pack\\data\\interactions.parquet'
products_path = 'C:\\Users\\rossim\\Desktop\\venchi_project\\venchi_jMLE_candidate_pack\\data\\products.txt'
sales_path = 'C:\\Users\\rossim\\Desktop\\venchi_project\\venchi_jMLE_candidate_pack\\data\\sales.parquet'

In [4]:
df = build_customer_dataset(accounts_path, interactions_path, products_path, sales_path)

In [5]:
df

,account_id,hct,staff,turnover_m_usd,brand_loyalty,timestamp,total_revenue_usd,interactions_last_6m,positive_interaction_rate,distinct_categories_bought
0,93963cf889dd72672a529096c97cae8b,50,18,89,8,2020-06-07 11:40:37,112118.0,1.0,0.500000,4.0
1,809d7aea9eacf339b2e35e3c8ae0a57c,74,6,71,8,2020-06-07 11:40:37,83850.0,0.0,0.000000,4.0
2,93189e2c4c7b1a2c7b16a24d5daa98a9,31,5,82,6,2020-06-07 11:40:37,77883.0,0.0,0.500000,4.0
3,fbcd0ff0529a3dd9b733884b30941297,21,3,98,3,2020-06-07 11:40:37,106749.0,0.0,0.666667,4.0
4,201e9991afe90c65e13b08b53fb695de,37,9,4,2,2020-06-07 11:40:37,14348.0,0.0,0.000000,2.0
...,...,...,...,...,...,...,...,...,...,...
12200,8ea4640392152934914886832f76051c,10,24,63,17,2020-01-31 18:22:15,73412.0,0.0,0.000000,4.0
12201,92c34ca373190ec1fbfecd476f3288a1,8,15,61,13,2020-01-31 18:22:15,84542.0,0.0,1.000000,4.0
12202,7b42a397eb158e09e4ea71127519777a,96,19,69,14,2020-01-31 18:22:15,87600.0,0.0,1.000000,4.0
12203,79ba0f4317496b5fecfd866b3ceede90,73,20,71,10,2020-01-31 18:22:15,74563.0,0.0,0.000000,4.0


### Pyspark

In [ ]:
from pyspark.sql import functions as F

# ==========================================
# 1. DATA CLEANING & STANDARDIZATION
# ==========================================
# The prompt notes free-text errors. We'll clean strings by lowercasing and trimming whitespace.
# We also drop rows where account_id is entirely missing, as they can't be tied to a customer.

accounts_clean = accounts_df.filter(F.col("account_id").isNotNull())

interactions_clean = interactions_df.filter(F.col("account_id").isNotNull()) \
    .withColumn("topic_clean", F.lower(F.trim(F.col("topic")))) \
    .withColumn("channel_clean", F.lower(F.trim(F.col("channel")))) \
    .withColumn("response_clean", F.lower(F.trim(F.col("response"))))

sales_clean = sales_df.filter(F.col("account_id").isNotNull())
products_clean = products_df.filter(F.col("product_id").isNotNull())


# ==========================================
# 2. KPI 1: Total Revenue Generated
# ==========================================
# Calculation: Join Sales with Products to get the price, then sum per account.

# Join sales with products to get pricing
sales_with_price = sales_clean.join(products_clean, on="product_id", how="left")

# Aggregate revenue per customer
kpi_revenue = sales_with_price.groupBy("account_id").agg(
    F.sum("price").alias("total_revenue")
)


# ==========================================
# 3. KPI 2: Sales Interactions in the Last 6 Months
# ==========================================
# Because we don't have a "today's date", we find the maximum date in the interactions dataset 
# and use that as the anchor to look back 6 months.

# Find the most recent date in the interactions dataset
max_date_row = interactions_clean.select(F.max("date").alias("latest_date")).collect()[0]
latest_date = max_date_row["latest_date"]

# Filter for interactions roughly related to "sales" and within the last 6 months (180 days)
# Using `contains` helps catch variations like "sales call", "pre-sales", etc.
sales_interactions_6m = interactions_clean.filter(
    (F.col("topic_clean").contains("sale")) & 
    (F.col("date") >= F.date_sub(F.lit(latest_date), 180))
)

# Aggregate count per customer
kpi_recent_sales_interactions = sales_interactions_6m.groupBy("account_id").agg(
    F.count("interaction_id").alias("sales_interactions_last_6m")
)


# ==========================================
# 4. KPI 3 & 4: Unique Products & Avg Interaction Duration
# ==========================================

# KPI 3: Unique Products Purchased (from Sales)
kpi_unique_products = sales_clean.groupBy("account_id").agg(
    F.countDistinct("product_id").alias("unique_products_purchased")
)

# KPI 4: Average Interaction Duration (from Interactions)
# We filter out any negative durations just in case there are severe data entry errors
kpi_avg_duration = interactions_clean.filter(F.col("duration_mins") >= 0).groupBy("account_id").agg(
    F.round(F.avg("duration_mins"), 2).alias("avg_interaction_duration_mins")
)


# ==========================================
# 5. MASTER JOIN (Building the Analytical Dataset)
# ==========================================
# We use a series of LEFT JOINs starting from the main Accounts table to ensure
# we don't lose any customers, even if they have no interactions or sales yet.

analytical_dataset = accounts_clean \
    .join(kpi_revenue, on="account_id", how="left") \
    .join(kpi_recent_sales_interactions, on="account_id", how="left") \
    .join(kpi_unique_products, on="account_id", how="left") \
    .join(kpi_avg_duration, on="account_id", how="left")

# ==========================================
# 6. NULL IMPUTATION
# ==========================================
# Left joins create Nulls for customers without sales/interactions. 
# We must fill these with 0 to ensure downstream Machine Learning models don't break.

analytical_dataset = analytical_dataset.fillna({
    "total_revenue": 0,
    "sales_interactions_last_6m": 0,
    "unique_products_purchased": 0,
    "avg_interaction_duration_mins": 0
})

# Final check of the schema and top rows
analytical_dataset.printSchema()
analytical_dataset.show(5)